# Tutorial 3: Backtesting

Validate the search pipeline with walk-forward backtesting. No look-ahead bias — each trial searches only past data.

In [ ]:
import numpy as np
from the_similarity import load, backtest, Config, FeatureStore

## 1. Generate Test Data

We need enough history for the backtester: at least `3 * window_size + window_size + forward_bars` bars.

In [ ]:
np.random.seed(42)

# Trending random walk with regime changes
n = 3000
returns = np.random.normal(0.0003, 0.012, n)

# Add regime changes: high vol periods
returns[500:700] *= 2.0   # vol spike
returns[1500:1700] *= 2.0
returns[2300:2500] *= 1.8

prices = 100 * np.exp(np.cumsum(returns))
history = load(prices)
print(f"History: {len(history)} bars")

## 2. Run Backtest

Each trial:
1. Picks a random query window
2. Searches only `history[:query_start]` (no future data)
3. Projects forward and compares to actual outcome

In [ ]:
report = backtest(
    history,
    window_size=60,
    forward_bars=30,
    n_trials=20,        # keep low for tutorial speed
    stride=3,
    seed=42,
)
report.summary()

## 3. Understanding the Metrics

- **Hit rate**: What fraction of trials predicted the correct direction? >50% = better than coin flip
- **MAE**: Mean absolute error of P50 vs actual returns at the forecast horizon
- **Calibration**: For percentile P, what fraction of actuals fell below P? Should equal P/100
- **CRPS**: Continuous Ranked Probability Score — jointly measures sharpness + calibration. Lower = better

In [ ]:
print(f"Hit rate: {report.hit_rate:.1%}")
print(f"MAE: {report.mean_error:.4f}")
print(f"CRPS: {report.crps:.4f}")
print(f"Valid trials: {report.n_valid_trials}/{len(report.trials)}")
print()

# Calibration check
print("Calibration (ideal: actual ≈ expected):")
for p, rate in sorted(report.calibration.items()):
    expected = p / 100
    status = "✓" if abs(rate - expected) < 0.10 else "✗"
    print(f"  P{p}: {rate:.0%} (expected {expected:.0%}) {status}")

## 4. Inspect Individual Trials

In [ ]:
for i, trial in enumerate(report.valid_trials[:5]):
    direction = "↑" if trial.directional_hit else "↓✗"
    print(
        f"Trial {i+1}: bars {trial.query_start}-{trial.query_end}, "
        f"{trial.n_matches} matches, "
        f"top_score={trial.top_match_score:.1f}, "
        f"P50_error={trial.p50_error:.4f} {direction}"
    )

## 5. Backtest with Caching

For large backtests, the FeatureStore avoids recomputing expensive Tier 2 methods on windows that have been seen before.

In [ ]:
import tempfile, os, time

db_path = os.path.join(tempfile.gettempdir(), "backtest_cache.db")
store = FeatureStore(db_path)

t0 = time.time()
report_cached = backtest(
    history,
    window_size=60,
    forward_bars=30,
    n_trials=20,
    stride=3,
    seed=42,
    feature_store=store,
)
print(f"Cached backtest: {time.time() - t0:.1f}s, cache entries: {store.size}")

# Clean up
store.clear()
os.unlink(db_path)

## 6. Comparing Configurations

Use backtesting to decide which configuration works best for your data.

In [ ]:
configs = {
    "DTW+Pearson only": Config(
        active_methods=["dtw", "pearson_warped"],
        tier2_candidates=0,
    ),
    "All 9 methods": Config(),
}

for name, cfg in configs.items():
    t0 = time.time()
    r = backtest(
        history,
        window_size=60,
        forward_bars=30,
        n_trials=15,
        config=cfg,
        stride=3,
        seed=42,
    )
    elapsed = time.time() - t0
    print(f"{name}: hit_rate={r.hit_rate:.0%}, CRPS={r.crps:.4f}, time={elapsed:.1f}s")